In [1]:
import pandas as pd
import matplotlib.pyplot as plt

SUMMARY_CSV  = "kbc177_project_summary.csv"
NETID        = "kbc177"
WOCProjectID = "llnl_hiop"

df = pd.read_csv(
    SUMMARY_CSV,
    sep=";",
    header=None,
    names=["project", "commit_hash", "author", "commit_time", "commit_message"],
    engine="python",
    on_bad_lines="warn",
)
df["commit_time"] = pd.to_datetime(df["commit_time"], unit="s")

print("Loaded", len(df), "commit rows.")
print("Projects found:", sorted(df["project"].unique()))


def build_monthly_timeseries(project_df):
    project_df = project_df.copy()
    project_df["Month"] = project_df["commit_time"].dt.to_period("M")
    counts = project_df.groupby("Month").size().reset_index(name="#Commits")
    full_range = pd.period_range(
        start=counts["Month"].min(),
        end=counts["Month"].max(),
        freq="M",
    )
    monthly = (
        counts.set_index("Month")
        .reindex(full_range, fill_value=0)
        .rename_axis("Month")
        .reset_index()
    )
    monthly["Month"] = monthly["Month"].astype(str)
    return monthly


monthly_by_project = {p: build_monthly_timeseries(sub)
                      for p, sub in df.groupby("project")}
monthly = monthly_by_project[WOCProjectID].copy()
monthly.to_csv(NETID + "_commits_timeseries_.csv", sep=";", index=False)
print("Saved", NETID + "_commits_timeseries_.csv")

plt.figure(figsize=(10, 5))
plt.plot(monthly["Month"], monthly["#Commits"], marker="o", linestyle="-")
plt.xlabel("Time (YYYY-MM)")
plt.ylabel("Number of Commits")
plt.title("Commit Activity: " + WOCProjectID)
plt.grid(True)
plt.xticks(rotation=45)
step = max(1, len(monthly) // 20)
plt.xticks(monthly["Month"][::step], rotation=45)
plt.tight_layout()
plt.savefig(NETID + "_timeseries_" + WOCProjectID + ".png", dpi=300)
plt.close()
print("Saved", NETID + "_timeseries_" + WOCProjectID + ".png")


def analyze_gaps(monthly):
    ts = monthly.copy()
    ts["Month"] = pd.PeriodIndex(ts["Month"], freq="M")
    gaps = []
    start = None
    for _, row in ts.iterrows():
        if row["#Commits"] == 0 and start is None:
            start = row["Month"]
        elif row["#Commits"] > 0 and start is not None:
            end = row["Month"] - 1
            length = (end - start).n + 1
            gaps.append((start, end, length))
            start = None
    if start is not None:
        end = ts["Month"].iloc[-1]
        length = (end - start).n + 1
        gaps.append((start, end, length))
    longest = max(gaps, key=lambda g: g[2]) if gaps else (None, None, 0)
    ls, le, ll = longest
    if le is not None:
        after = int(ts.loc[ts["Month"] > le, "#Commits"].sum())
    else:
        after = 0
    num_gaps_ge_3 = sum(1 for _, _, l in gaps if l >= 3)
    return gaps, longest, num_gaps_ge_3, after


gaps, longest, num_gaps_ge_3, commits_after = analyze_gaps(monthly)
longest_start, longest_end, longest_len = longest

print("Gaps:")
for gs, ge, gl in gaps:
    print("  ", gs, "->", ge, "(", gl, "months )")
print("Longest gap:", longest_start, "->", longest_end, "(", longest_len, "months )")
print("Commits after longest gap:", commits_after)
print("Gaps >= 3 months:", num_gaps_ge_3)


def classify_pattern(monthly):
    counts = monthly["#Commits"].astype(int).values
    n = len(counts)
    if n < 4 or counts.sum() == 0:
        return "irregular"
    fq = counts[: n // 4].mean()
    lq = counts[-n // 4:].mean()
    mid = counts[n // 4 : -n // 4]
    mm = mid.mean() if len(mid) else 0
    if lq > 2 * fq and lq > mm:
        return "rising"
    if fq > 2 * lq and fq > mm:
        return "declining"
    if fq > 1.5 * mm and lq > 1.5 * mm:
        return "U-shaped"
    if num_gaps_ge_3 >= 2:
        return "cyclical"
    mean = counts.mean()
    if mean > 0 and counts.std() < 0.5 * mean:
        return "steady"
    return "irregular"


pattern = classify_pattern(monthly)
print("ActivityPattern (heuristic):", pattern)

project_df = df[df["project"] == WOCProjectID]
stats = pd.DataFrame([{
    "project":                WOCProjectID,
    "number of commits":      project_df.shape[0],
    "number of authors":      project_df["author"].nunique(),
    "max time":               project_df["commit_time"].max(),
    "min time":               project_df["commit_time"].min(),
    "number of stars":        None,
    "number of forks":        None,
    "last commit date":       project_df["commit_time"].max(),
    "LongestGapStart":        str(longest_start) if longest_start is not None else "",
    "LongestGapEnd":          str(longest_end)   if longest_end   is not None else "",
    "LongestGapLength":       longest_len,
    "NumCommitsAfterLastGap": commits_after,
    "NumberOfGapsInTimeline": num_gaps_ge_3,
    "ActivityPattern":        pattern,
}])
stats.to_csv(NETID + "_project_stats.csv", index=False)
print("Saved", NETID + "_project_stats.csv")
print(stats.T)

Loaded 50827 commit rows.
Projects found: ['barbagroup_aeropython', 'brian-team_brian2genn', 'helmchenlabsoftware_cascade', 'insightsoftwareconsortium_simpleitk-notebooks', 'llnl_hiop', 'magic-sph_magic', 'pulp-platform_axi', 'pymor_pymor', 'stan-dev_bayesplot', 'yeatmanlab_afq-browser']
Saved kbc177_commits_timeseries_.csv
Saved kbc177_timeseries_llnl_hiop.png
Gaps:
   2016-10 -> 2016-10 ( 1 months )
   2017-03 -> 2017-03 ( 1 months )
   2017-06 -> 2017-07 ( 2 months )
   2017-09 -> 2017-09 ( 1 months )
   2019-01 -> 2019-01 ( 1 months )
   2019-03 -> 2019-03 ( 1 months )
   2019-05 -> 2019-06 ( 2 months )
   2019-09 -> 2019-09 ( 1 months )
   2024-04 -> 2024-04 ( 1 months )
   2024-07 -> 2024-07 ( 1 months )
Longest gap: 2017-06 -> 2017-07 ( 2 months )
Commits after longest gap: 3620
Gaps >= 3 months: 0
ActivityPattern (heuristic): irregular
Saved kbc177_project_stats.csv
                                          0
project                           llnl_hiop
number of commits        

Interpretation — llnl_hiop (kbc177)
The monthly commit time series (Sep 2016 – Nov 2025, 3,737 commits, 91 authors) shows continuous, bursty activity across the project's entire 9-year history. There is no clear rising, declining, U-shaped, steady, or cyclical trend of monthly volumes fluctuate unpredictably, so the heuristic classifier labels the pattern irregular. The longest inactivity gap was only 2 months (2017-06 → 2017-07), and 3,620 commits occurred after that gap. Number of gaps of at least 3 months: 0. The project is best described as actively maintained with irregular bursts and no significant dormancy, rather than abandoned or intermittent.